# LLC Additivity on CMSP — Google Colab

**Setup**: Runtime → Change runtime type → **T4 GPU** (or better)

This notebook runs the full pipeline: train CMSP models, estimate data-restricted LLCs, and compute additivity defects.

## 0. Install & Setup

In [ ]:
# Mount Google Drive (optional — for persisting results across sessions)
SAVE_TO_DRIVE = True  # Set False to save locally (lost on runtime disconnect)

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    RESULTS_ROOT = '/content/drive/MyDrive/llc_cmsp_results'
else:
    RESULTS_ROOT = '/content/results'

In [ ]:
%%bash
# Clone repos and install dependencies
cd /content

# devinterp (LLC estimation library)
if [ ! -d devinterp ]; then
    git clone https://github.com/timaeus-research/devinterp.git
fi
pip install -e ./devinterp -q

# Additional deps
pip install pyyaml tqdm pandas matplotlib -q

In [ ]:
# Upload the llc-modularity project.
# Option A: Clone from your repo (if you've pushed it)
# !git clone https://github.com/YOUR_USER/llc-modularity.git /content/llc-modularity

# Option B: Upload from local machine
# Use the file browser on the left to upload the llc-modularity/ folder,
# or zip it and upload:
#   from google.colab import files
#   files.upload()  # upload llc-modularity.zip
#   !unzip llc-modularity.zip -d /content/

# Option C: If you're running this notebook FROM the repo, copy it in place
import os
if not os.path.exists('/content/llc-modularity/src'):
    print('ERROR: llc-modularity/src not found at /content/llc-modularity/')
    print('Upload the project using one of the options above, then re-run this cell.')
else:
    print('llc-modularity project found.')

In [ ]:
import sys
sys.path.insert(0, '/content/llc-modularity')

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    DEVICE = 'cuda'
else:
    print('WARNING: No GPU detected. LLC estimation will be very slow.')
    DEVICE = 'cpu'

In [ ]:
# Quick import check
from src.data import generate_cmsp_batch, make_subtask_indices, CMSPDataset
from src.model import make_mlp, count_parameters
from src.train import train_cmsp
from src.llc_estimation import estimate_llc, estimate_subtask_llcs
from src.additivity import compute_additivity_defect, compute_full_additivity_defect, summarize_results
from src.data import make_subtask_dataloaders, make_union_dataloaders, code_to_str
print('All imports OK')

## 1. Configuration

Edit the config dict below to control the experiment. Defaults match the paper's standard setup.

In [ ]:
import os
from pathlib import Path

# ============================================================
# EXPERIMENT CONFIG — edit this cell
# ============================================================

EXPERIMENT = 'B'  # 'A' = independent atomics, 'B' = composite curriculum

config = dict(
    # Task
    m=4,            # number of atomic subtasks
    k=4,            # bits per subtask
    n=16,           # total task bits (m * k)

    # Model
    width=128,
    depth=3,
    activation='relu',
    use_layernorm=False,

    # Training
    samples_per_task=2000,
    lr=1e-3,
    dtype='float32',
    device=DEVICE,
    eval_every=500,
    checkpoint_every=0,
    checkpoint_steps=[],
)

if EXPERIMENT == 'A':
    config['task_codes'] = [[0], [1]]  # independent atomics only
    config['steps'] = 100_000
elif EXPERIMENT == 'B':
    config['task_codes'] = [[0], [1], [2], [3], [0, 1, 2, 3]]  # atomics + composite
    config['steps'] = 200_000
else:
    raise ValueError(f'Unknown experiment: {EXPERIMENT}')

# LLC estimation hyperparameters
LLC_CONFIG = dict(
    num_chains=10,
    num_draws=500,
    num_burnin_steps=100,
    num_steps_bw_draws=1,
    learning_rate=1e-4,
    localization=0.0,
    seed=42,
)

LLC_SAMPLES_PER_CODE = 2000
LLC_BATCH_SIZE = 256

# Seeds to run
SEEDS = [0, 1, 2]

print(f'Experiment {EXPERIMENT}: codes={config["task_codes"]}, '
      f'width={config["width"]}, depth={config["depth"]}, steps={config["steps"]}')
print(f'Seeds: {SEEDS}')
print(f'LLC: {LLC_CONFIG["num_chains"]} chains x {LLC_CONFIG["num_draws"]} draws')

## 2. Train Models

In [ ]:
trained_models = {}

for seed in SEEDS:
    print(f'\n{"="*60}')
    print(f'Training seed {seed}')
    print(f'{"="*60}')

    cfg = {**config, 'seed': seed}
    save_dir = os.path.join(RESULTS_ROOT, f'exp_{EXPERIMENT}', f'seed_{seed}')
    os.makedirs(save_dir, exist_ok=True)

    results = train_cmsp(cfg, save_dir=save_dir, verbose=True)
    trained_models[seed] = results

    print(f'  Parameters: {results["n_parameters"]}')
    print('  Final losses:')
    for name, loss in results['final_subtask_losses'].items():
        print(f'    {name}: {loss:.6f}')

### 2a. Training Curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(SEEDS), figsize=(6*len(SEEDS), 4), squeeze=False)

for i, seed in enumerate(SEEDS):
    ax = axes[0, i]
    res = trained_models[seed]
    eval_steps = res['eval_steps']
    for name, losses in res['subtask_losses'].items():
        ax.plot(eval_steps[:len(losses)], losses, label=name)
    ax.set_xlabel('Step')
    ax.set_ylabel('Loss')
    ax.set_title(f'Seed {seed}')
    ax.set_yscale('log')
    ax.legend(fontsize=8)

plt.suptitle('Per-Subtask Training Loss', fontsize=14)
plt.tight_layout()
plt.show()

## 3. LLC Estimation

For each trained model, estimate the LLC using data restricted to each subtask.

In [ ]:
import pickle

m = config['m']
k = config['k']
n = config['n']
subtask_indices = make_subtask_indices(m, k)
task_codes = config['task_codes']
dtype = torch.float32

all_llc_results = {}

for seed in SEEDS:
    print(f'\n{"="*60}')
    print(f'LLC estimation — seed {seed}')
    print(f'{"="*60}')

    model = trained_models[seed]['model']

    # Per-code dataloaders
    per_code_loaders = make_subtask_dataloaders(
        n=n, m=m, subtask_indices=subtask_indices,
        task_codes=task_codes,
        samples_per_code=LLC_SAMPLES_PER_CODE,
        batch_size=LLC_BATCH_SIZE,
        device=DEVICE, dtype=dtype,
        seed=LLC_CONFIG['seed'],
    )

    # Union dataloaders (pairwise atomics + all atomics combined)
    atomic_codes = [c for c in task_codes if len(c) == 1]
    union_groups = {}
    for i in range(len(atomic_codes)):
        for j in range(i + 1, len(atomic_codes)):
            label = f'{code_to_str(atomic_codes[i])}\u222a{code_to_str(atomic_codes[j])}'
            union_groups[label] = [atomic_codes[i], atomic_codes[j]]
    if len(atomic_codes) > 2:
        all_label = '\u222a'.join(code_to_str(c) for c in atomic_codes)
        union_groups[all_label] = atomic_codes

    if union_groups:
        union_loaders = make_union_dataloaders(
            n=n, m=m, subtask_indices=subtask_indices,
            code_groups=union_groups,
            samples_per_code=LLC_SAMPLES_PER_CODE,
            batch_size=LLC_BATCH_SIZE,
            device=DEVICE, dtype=dtype,
            seed=LLC_CONFIG['seed'],
        )
        per_code_loaders.update(union_loaders)

    # Run LLC estimation
    llc_results = estimate_subtask_llcs(
        model=model,
        subtask_dataloaders=per_code_loaders,
        device=DEVICE,
        verbose=True,
        **LLC_CONFIG,
    )

    all_llc_results[seed] = llc_results

    # Save
    save_dir = os.path.join(RESULTS_ROOT, f'exp_{EXPERIMENT}', f'seed_{seed}')
    with open(os.path.join(save_dir, 'llc_results.pkl'), 'wb') as f:
        pickle.dump(llc_results, f)

print('\nLLC estimation complete for all seeds.')

### 3a. LLC Summary

In [ ]:
import pandas as pd

rows = []
for seed, llc_res in all_llc_results.items():
    for name, r in llc_res.items():
        rows.append({'seed': seed, 'subtask': name,
                     'llc_mean': r['llc_mean'], 'llc_std': r['llc_std']})

llc_df = pd.DataFrame(rows)
summary = llc_df.groupby('subtask').agg(
    llc_avg=('llc_mean', 'mean'),
    llc_sem=('llc_mean', 'std'),
).sort_index()
print(summary.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(range(len(summary)), summary['llc_avg'], yerr=summary['llc_sem'],
       capsize=5, alpha=0.8, color='steelblue')
ax.set_xticks(range(len(summary)))
ax.set_xticklabels(summary.index, rotation=45, ha='right')
ax.set_ylabel(r'LLC ($\hat{\lambda}$)')
ax.set_title(f'Experiment {EXPERIMENT}: LLC per Data Restriction (avg over {len(SEEDS)} seeds)')
plt.tight_layout()
plt.show()

### 3b. SGLD Trace Diagnostics

In [ ]:
# Check SGLD mixing for first seed
seed0 = SEEDS[0]
llc_res = all_llc_results[seed0]
subtasks_to_plot = [k for k in sorted(llc_res.keys()) if '\u222a' not in k][:4]

fig, axes = plt.subplots(1, len(subtasks_to_plot), figsize=(5*len(subtasks_to_plot), 4))
if len(subtasks_to_plot) == 1:
    axes = [axes]

for ax, name in zip(axes, subtasks_to_plot):
    trace = llc_res[name].get('loss_trace')
    if trace is not None:
        import numpy as np
        trace = np.array(trace) if not isinstance(trace, np.ndarray) else trace
        for chain_trace in trace:
            ax.plot(chain_trace, alpha=0.4, linewidth=0.8)
    ax.set_title(name)
    ax.set_xlabel('Draw')
    ax.set_ylabel('Loss')

plt.suptitle(f'SGLD Loss Traces (seed {seed0})', fontsize=14)
plt.tight_layout()
plt.show()

## 4. Additivity Defects

In [ ]:
# Compute pairwise defects for each seed
all_defect_dfs = []

for seed, llc_res in all_llc_results.items():
    available = set(llc_res.keys())
    atomics = sorted([k for k in available
                      if k.startswith('{') and ',' not in k and '\u222a' not in k])

    triples = []
    for i in range(len(atomics)):
        for j in range(i + 1, len(atomics)):
            union_key = f'{atomics[i]}\u222a{atomics[j]}'
            if union_key in available:
                triples.append((atomics[i], atomics[j], union_key))

    if triples:
        df = compute_additivity_defect(llc_res, triples)
        df['seed'] = seed
        all_defect_dfs.append(df)

if all_defect_dfs:
    defect_df = pd.concat(all_defect_dfs, ignore_index=True)
    agg = defect_df.groupby(['T1', 'T2']).agg(
        delta_avg=('delta', 'mean'),
        delta_sem=('delta', 'std'),
    )
    print('Pairwise Additivity Defects (averaged over seeds):')
    print(agg.to_string())
else:
    print('No pairwise defects could be computed.')

In [ ]:
if all_defect_dfs:
    pairs = defect_df.groupby(['T1', 'T2'])['delta'].agg(['mean', 'std']).reset_index()
    labels = [f"{row['T1']}+{row['T2']}" for _, row in pairs.iterrows()]

    fig, ax = plt.subplots(figsize=(10, 5))
    colors = ['#2196F3' if d >= 0 else '#F44336' for d in pairs['mean']]
    ax.bar(range(len(labels)), pairs['mean'], yerr=pairs['std'],
           capsize=5, color=colors, alpha=0.8)
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha='right')
    ax.set_ylabel(r'Additivity defect $\delta$')
    ax.set_title(f'Experiment {EXPERIMENT}: Pairwise LLC Additivity Defects')
    ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
    plt.tight_layout()
    plt.show()

### 4a. Full Composite Defect (Experiment B only)

In [ ]:
if EXPERIMENT == 'B':
    full_defects = []
    atomic_names = [code_to_str([i]) for i in range(config['m'])]
    # Find the all-atomics union key
    composite_key = '\u222a'.join(atomic_names)

    for seed, llc_res in all_llc_results.items():
        if composite_key in llc_res:
            r = compute_full_additivity_defect(llc_res, atomic_names, composite_key)
            r['seed'] = seed
            full_defects.append(r)
            print(f"Seed {seed}: sum_atomic={r['sum_atomic_llc']:.3f}, "
                  f"composite={r['composite_llc']:.3f}, "
                  f"delta={r['delta']:.3f} +/- {r['delta_std']:.3f} "
                  f"(z={r['delta_zscore']:.2f})")

    if full_defects:
        deltas = [r['delta'] for r in full_defects]
        import numpy as np
        print(f'\nFull composite defect: {np.mean(deltas):.3f} +/- {np.std(deltas):.3f} '
              f'(n={len(deltas)} seeds)')
    else:
        print(f'Composite key "{composite_key}" not found in LLC results.')
        print(f'Available keys: {list(all_llc_results[SEEDS[0]].keys())}')
else:
    print('Full composite defect only computed for Experiment B.')

## 5. Save All Results

In [ ]:
save_path = os.path.join(RESULTS_ROOT, f'exp_{EXPERIMENT}', 'summary.pkl')
os.makedirs(os.path.dirname(save_path), exist_ok=True)

summary_data = {
    'config': config,
    'llc_config': LLC_CONFIG,
    'seeds': SEEDS,
    'llc_df': llc_df,
    'defect_df': defect_df if all_defect_dfs else None,
}

with open(save_path, 'wb') as f:
    pickle.dump(summary_data, f)

print(f'Summary saved to {save_path}')
if SAVE_TO_DRIVE:
    print('Results are on Google Drive and will persist across sessions.')

---

## Appendix: Quick Single-Seed Test Run

Use this to validate the pipeline quickly with a small model before committing to a full run.

In [ ]:
# Uncomment and run this cell for a quick smoke test (~2 min on GPU)
"""
quick_config = dict(
    m=2, k=3, n=6,
    width=32, depth=2,
    activation='relu', use_layernorm=False,
    task_codes=[[0], [1]],
    samples_per_task=500,
    steps=5000, lr=1e-3,
    seed=0, dtype='float32',
    device=DEVICE, eval_every=500,
    checkpoint_every=0, checkpoint_steps=[],
)

res = train_cmsp(quick_config, verbose=True)
print('Final losses:', res['final_subtask_losses'])

subtask_indices = make_subtask_indices(2, 3)
loaders = make_subtask_dataloaders(
    n=6, m=2, subtask_indices=subtask_indices,
    task_codes=[[0], [1]], samples_per_code=500,
    batch_size=128, device=DEVICE, dtype=torch.float32, seed=42,
)
union_loaders = make_union_dataloaders(
    n=6, m=2, subtask_indices=subtask_indices,
    code_groups={'{0}\u222a{1}': [[0], [1]]},
    samples_per_code=500, batch_size=128,
    device=DEVICE, dtype=torch.float32, seed=42,
)
loaders.update(union_loaders)

llc_res = estimate_subtask_llcs(
    model=res['model'], subtask_dataloaders=loaders,
    device=DEVICE, num_chains=3, num_draws=50,
    num_burnin_steps=20, learning_rate=1e-4, seed=42, verbose=True,
)

df = compute_additivity_defect(llc_res, [('{0}', '{1}', '{0}\u222a{1}')])
print(summarize_results(llc_res, df))
"""